# Bands and projected DOS of an SCF run

Replaces the per-run `pdos_bands.ipynb`. Needs `projwfc.x` output (`<prefix>.pdos_atm#...`) in `SCF_DIR`; so far
the no-SOC SCF runs are the ones with PDOS.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "modules/axion.py").exists())
DATA = ROOT / "data"
sys.path.insert(0, str(ROOT))
from modules.qe_read_file import read_bands_out, read_kpath, fermi_energy, read_pdos_by_element

In [ ]:
PREFIX = "SrTiO3"
SCF_DIR = DATA / "SrTiO3/base/wo_soc/output/181639bc889"     # holds <prefix>.scf.out and the pdos_atm# files
BANDS_OUT = SCF_DIR / "SrTiO3.bands.out"                      # pw.x bands output; None to leave the panel empty
BANDS_IN = SCF_DIR / "SrTiO3.bands.in"
ORBITALS = None                  # e.g. [("Ir", "d"), ("O", "p")]; None plots every (element, orbital)
E_WINDOW = (-10.0, 6.0)          # eV relative to E_F

In [ ]:
e_fermi = fermi_energy(SCF_DIR / f"{PREFIX}.scf.out")
pdos = read_pdos_by_element(SCF_DIR, PREFIX)
if not pdos:
    raise FileNotFoundError(f"no {PREFIX}.pdos_atm#* files in {SCF_DIR}")

fig, (ax_b, ax_d) = plt.subplots(1, 2, figsize=(12, 8), gridspec_kw={"width_ratios": [2, 1]}, sharey=True)
if BANDS_OUT:
    k_cart, bands = read_bands_out(BANDS_OUT)
    distance = np.r_[0, np.cumsum(np.linalg.norm(np.diff(k_cart, axis=0), axis=1))]
    for band in bands:
        ax_b.plot(distance, band - e_fermi, color="black", lw=0.6)
    if BANDS_IN:
        _, ticks = read_kpath(BANDS_IN)
        ax_b.set_xticks([distance[i] for i, _ in ticks], [label for _, label in ticks])
        for i, _ in ticks:
            ax_b.axvline(distance[i], color="gray", lw=0.4)
    ax_b.set_xlim(distance[0], distance[-1])
ax_b.set(ylabel=r"$E - E_F$ (eV)", ylim=E_WINDOW, title=str(SCF_DIR.relative_to(DATA)))
for key in ORBITALS or sorted(pdos):
    energy, dos = pdos[key]
    ax_d.fill_betweenx(energy - e_fermi, dos, alpha=0.35, label=f"{key[0]} {key[1]}")
ax_d.set(xlabel="PDOS", title="projected DOS")
ax_d.legend(frameon=False, fontsize=8)
for ax in (ax_b, ax_d):
    ax.axhline(0, color="red", lw=0.6, ls=":")
plt.tight_layout()
plt.show()